# Four ways a posterior can be a restatement of your assumptions

A posterior can look like a finding and be none of them. It can be the prior, barely moved,
because the data has nothing to say about that parameter. It can be one point on a ridge, where
a hundred other parameter combinations fit equally well. It can be an artifact of the prior
being nonsense on the outcome scale, which nobody checked before fitting. And it can be one of
sixteen defensible analyses — the one that got run.

Each of those has a check here, and each returns a number rather than a reassurance.

Three questions about one fit. *Is the posterior geometry healthy?* (`weak_identification`:
ridges, condition number, unlearned and saturated parameters.) *Did the data move the prior?*
(`learning`: contraction, overlap, shift.) *Does the prior imply sensible responses before
any data?* (`prior_predictive`.) And a fourth about many fits: *does the conclusion survive
the analyst's choices?* (`specification_curve`.)

In [ ]:
import numpy as np

from axiom.core import (
    Add, Const, Data, Likelihood, ModelSpec, Mul, Param, Posterior, Prior, Unsupported, dimensionless,
)
from axiom.diagnose import (
    ExpectedSign, FitSettings, LearningReport, ParameterLearning, PriorPredictive, SpecCurve,
    SpecCurveSummary, SpecificationAxis, SpecOption, SpecRow, WeakIdReport, apply_option,
    bhattacharyya, default_estimand, learning, prior_moments, prior_predictive, realized_point,
    specification_curve, weak_identification,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import LogisticKernel, fit

from axiom.display import enable, table
from axiom.viz import spec_curve_plot

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, compare, dumbbell, heat, mark_x

enable();  # every axiom result renders itself from here on

DL = dimensionless()

## Weak identification from posterior geometry

`prior_moments` gives the closed-form mean and sd of any `Prior` family (hyper-parameters may
be resolved from a mapping). `weak_identification` takes a `Posterior` (or a `FitResult`, in
which case it restricts to the structural parameters) and reports the correlation matrix, the
condition number of the standardized covariance, pairs above `rho_threshold` (the `k–s–beta`
ridge of a Hill surface), the prior/posterior sd ratio per parameter, and which parameters are
unlearned or saturated at a bound.

In [ ]:
print("normal(1, 3):", prior_moments(Prior(family="normal", hyper={"mu": 1, "sigma": 3})))
print("lognormal(0, 0.5):", np.round(prior_moments(Prior(family="lognormal", hyper={"mu": 0, "sigma": 0.5})), 4))
print("resolved hyper:", prior_moments(Prior(family="normal", hyper={"mu": "m", "sigma": 1}), {"m": 3.0}))

x = Data(name="x", dimension=DL)
a = Param(name="a", dimension=DL, prior=Prior(family="normal", hyper={"mu": 0, "sigma": 2}))
b = Param(name="b", dimension=DL, prior=Prior(family="normal", hyper={"mu": 0, "sigma": 2}))
r = Param(name="r", dimension=DL, prior=Prior(family="beta", hyper={"alpha": 2, "beta": 2}))
s = Param(name="sigma", dimension=DL, prior=Prior(family="halfnormal", hyper={"sigma": 1}))
toy = ModelSpec(
    name="toy",
    mean=Add(terms=(a, Mul(factors=(b, x)), Mul(factors=(r, Const(value=0.0, dimension=DL))))),
    outcome=Data(name="y", dimension=DL),
    likelihood=Likelihood(family="normal", scale="sigma"),
    parameters=(a, b, r, s),
)

rng = np.random.default_rng(0)
n = 4000
ad = rng.normal(0.0, 0.2, n)
post = Posterior({"a": ad[None], "b": (0.99 * ad + rng.normal(0, 0.02, n))[None], "r": rng.beta(2, 2, n)[None], "sigma": (np.abs(ad) + 0.1)[None]})
rep = weak_identification(post, toy, rho_threshold=0.9)
assert isinstance(rep, WeakIdReport)
print("ridge pairs:", [(p, q, round(rho, 3)) for p, q, rho in rep.high_pairs])
print("condition number:", round(rep.condition_number, 1), "| sd ratios:", {k: round(v, 2) for k, v in rep.sd_ratio.items()})
print("unlearned:", rep.unlearned, "| saturated:", rep.saturated, "| passed:", rep.passed)

In [ ]:
names = list(rep.parameters)
fig = heat(
    np.asarray(rep.correlation), names, names,
    diverging=True, zmid=0.0,
    colorbar_title="corr",
    title="A ridge, drawn",
    subtitle=f"posterior correlation between parameters; condition number {rep.condition_number:.0f}",
    height=320,
)
caption(fig, "The two dark cells are a pair of parameters the data cannot separate: move one, "
             "move the other, and the fit is equally good. A posterior summary reports each of "
             "them with a respectable interval and never mentions that only their combination "
             "is pinned down.")

## Did the data move the prior?

`bhattacharyya` is the overlap coefficient of two normals. `learning` draws from the prior
(through `draw_prior`) and, per parameter, reports the contraction `1 − sd_post²/sd_prior²`,
the Bhattacharyya overlap, and the shift in prior sds; parameters with contraction below
`threshold` are flagged `prior_dominated`.

In [ ]:
print("same:", bhattacharyya(0, 1, 0, 1), "| far:", round(bhattacharyya(0, 1, 10, 1), 8), "| wider:", round(bhattacharyya(0, 1, 0, 2), 4))
learned = Posterior({"a": rng.normal(1.0, 0.2, (1, n)), "b": rng.normal(0.0, 2.0, (1, n)), "sigma": np.abs(rng.normal(0, 1, (1, n)))})
lr = learning(toy, learned, n_prior=4000, seed=1, threshold=0.1)
assert isinstance(lr, LearningReport)
rows = []
for p in lr.parameters:
    assert isinstance(p, ParameterLearning)
    rows.append([p.name, f"{p.contraction:.3f}", f"{p.overlap:.3f}", f"{p.shift:.3f}",
                 str(p.prior_dominated)])
table(rows, headers=("parameter", "contraction", "overlap", "shift", "prior dominated"))
print("prior-dominated:", lr.prior_dominated, "| passed:", lr.passed)

In [ ]:
fig = compare(
    [f"{p.name}{'  (prior-dominated)' if p.prior_dominated else ''}" for p in lr.parameters],
    [p.contraction for p in lr.parameters],
    highlight=next((f"{p.name}  (prior-dominated)" for p in lr.parameters if p.prior_dominated), None),
    value_fmt="{:.2f}",
    title="Which parameters the data actually moved",
    subtitle="contraction = 1 − posterior variance / prior variance; zero means the prior came back unchanged",
    x_title="contraction",
)
caption(fig, "A contraction near zero is a parameter whose posterior is its prior wearing a "
             "different name. Reporting it as a result is reporting an assumption, and this "
             "is the check that names which ones.")

## Prior predictive of a surface

`prior_predictive` pushes prior draws through `Surface.forward` on the observed doses and
summarizes the implied response range and each treatment's contribution, flagging the share of
prior mass with the wrong `ExpectedSign` or a magnitude beyond `magnitude_factor` × the
outcome sd. `passed` is `share_flagged <= tolerance`.

In [ ]:
world = surface_world(n_units=3, n_periods=10, treatments=("a", "b"), doses=DosePlan(scale=50.0, zero_fraction=0.1), intercept="shared", noise_sd=0.3, seed=3)
sign: ExpectedSign = "positive"
pp = prior_predictive(world.spec, world.panel, n=60, seed=4, expected_sign=sign, magnitude_factor=10.0)
assert isinstance(pp, PriorPredictive)
print("response range:", pp.response_range)
table(
    [
        [t, f"{pp.contribution[t].mean:.2f}", f"{pp.share_wrong_sign[t]:.2f}",
         f"{pp.share_implausible_magnitude[t]:.2f}"]
        for t in pp.treatments
    ],
    headers=("treatment", "contribution mean", "share wrong sign", "share implausible"),
)
print("share flagged:", pp.share_flagged, "| passed:", pp.passed)
strict = prior_predictive(world.spec, world.panel, n=60, seed=4, magnitude_factor=1e-9)
print("with an impossible magnitude threshold every draw is flagged:", strict.share_flagged, strict.passed)

## The specification curve

A `SpecificationAxis` names one analyst choice and its `SpecOption`s; each option is *data*
(a `spec_update` over `SurfaceSpec` fields, a `fit_update` over `FitSettings`), so the curve
serializes. `apply_option` shows what one option does to the base spec;
`default_estimand` is the per-unit cumulative contrast at the mean positive dose;
`realized_point` realizes it on one fit (or returns a typed failure).
`specification_curve` fits the cartesian product (capped by `max_specs`) and every row is a
`SpecRow` — a point with its interval, or a recorded failure.

In [ ]:
small = surface_world(n_units=3, n_periods=10, treatments=("a",), intercept="shared", doses=DosePlan(scale=50.0, zero_fraction=0.2), noise_sd=0.2, seed=11)
axes = (
    SpecificationAxis(name="kernel", options=(
        SpecOption(label="hill"),
        SpecOption(label="logistic", spec_update={"kernels": {"a": LogisticKernel(reference_dose=50.0)}}),
    )),
    SpecificationAxis(name="intercept_scale", options=(
        SpecOption(label="tight", spec_update={"intercept_scale": 0.5}),
        SpecOption(label="wide", spec_update={"intercept_scale": 5.0}, fit_update={"draws": 50}),
    )),
)
spec_l, settings_l = apply_option(small.spec, FitSettings(draws=40), axes[0].options[1])
print("logistic option ->", spec_l.kernel_of("a").name, "| settings:", settings_l)
est = default_estimand(small.spec, small.panel)
print("default estimand:", est.name, "dose", round(est.intervention.doses["a"], 2), "window", est.window.start, "-", est.window.stop)
base_fit = fit(small.spec, small.panel, backend="laplace", draws=40, chains=1, seed=1)
pt = realized_point(est, base_fit, definition="eti", mass=0.9)
assert not isinstance(pt, Unsupported)
print("realized on the base fit:", round(pt.result.summary.mean, 3), pt.result.summary.interval)

In [ ]:
curve = specification_curve(small.spec, small.panel, axes, draws=40, chains=1, seed=3)
assert isinstance(curve, SpecCurve)
rows = []
for row in curve.rows:
    assert isinstance(row, SpecRow)
    rows.append([str(row.labels), f"{row.estimate:.3f}",
                 f"[{row.interval.lower:.3f}, {row.interval.upper:.3f}]", str(row.excludes_zero)])
table(rows, headers=("specification", "estimate", "interval", "excludes 0"))
summary = curve.summary()
assert isinstance(summary, SpecCurveSummary)
print(f"median {summary.median:.3f}, IQR [{summary.iqr_lower:.3f}, {summary.iqr_upper:.3f}], share excluding zero {summary.share_excluding_zero}")
print("fitted", curve.n_total, "of", curve.n_total + curve.n_dropped, "| round-trips:", SpecCurve.from_json(curve.to_json()) == curve)

In [ ]:
spec_curve_plot(curve)

Every point on that curve is an analysis somebody could have run and defended. The one that
*was* run is one of them, and the honest summary is the spread — the median, the interquartile
range, and the share of specifications whose interval excludes zero — rather than the point
that happened to be produced first.

The specification axes are `Spec` data, not code, so the curve serializes into the analysis and
a reviewer can add an axis and re-run it.

## What this bought you

Four checks that each turn "the result looks reasonable" into something falsifiable: a
correlation matrix that names the ridge, a contraction per parameter that names what the data
did not move, a prior predictive that catches an implausible prior before fitting, and a
specification curve that reports the spread across the analyses you did not publish.